# Linear Regression
**Mục tiêu**: Triển khai mô hình Hồi quy Tuyến tính (Linear Regression) từ đầu bằng PyTorch để dự đoán giá mua xe, sau đó đánh giá hiệu năng trên tập Test.

## 1. Import thư viện & Cấu hình

In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# === Đường dẫn dữ liệu ===
DATA_DIR  = "../data/preprocessed"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
VAL_CSV   = os.path.join(DATA_DIR, "val.csv")
TEST_CSV  = os.path.join(DATA_DIR, "test.csv")

# === Siêu tham số (Hyperparameters) ===
SEED       = 42
BATCH_SIZE = 64
LR         = 1e-2     # Learning rate cho Linear Regression cao hơn MLP
EPOCHS     = 200

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

Device: cpu


## 2. Đọc dữ liệu đã tiền xử lý

In [2]:
def read_xy(path):
    """Đọc file CSV, tách features (X) và target (y)."""
    df = pd.read_csv(path)
    df = df.dropna(how="all")
    X = df.iloc[:, :-1].values.astype(np.float32)
    y = df.iloc[:, -1].values.astype(np.float32).reshape(-1, 1)
    return df, X, y

df_train, X_train, y_train = read_xy(TRAIN_CSV)
df_val,   X_val,   y_val   = read_xy(VAL_CSV)
df_test,  X_test,  y_test  = read_xy(TEST_CSV)

print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)
display(df_train.head())

Train: (400, 5) (400, 1)
Val:   (50, 5) (50, 1)
Test:  (50, 5) (50, 1)


## 3. Chuẩn bị DataLoader
Chuẩn hóa Z-score trên tập Train, rồi áp dụng cho Validation và Test.

In [3]:
# Tính mean/std từ tập Train
mean = X_train.mean(axis=0, keepdims=True)
std  = X_train.std(axis=0, keepdims=True)
std[std == 0.0] = 1.0  # Tránh chia cho 0

# Chuẩn hóa Z-score
X_train_n = (X_train - mean) / std
X_val_n   = (X_val   - mean) / std
X_test_n  = (X_test  - mean) / std

# Chuyển sang PyTorch Tensor
tX_train = torch.from_numpy(X_train_n)
ty_train = torch.from_numpy(y_train)
tX_val   = torch.from_numpy(X_val_n)
ty_val   = torch.from_numpy(y_val)
tX_test  = torch.from_numpy(X_test_n)
ty_test  = torch.from_numpy(y_test)

# Tạo DataLoader
train_ds = TensorDataset(tX_train, ty_train)
val_ds   = TensorDataset(tX_val, ty_val)
test_ds  = TensorDataset(tX_test, ty_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Số features: {tX_train.shape[1]}")

Số features: 5


## 4. Định nghĩa mô hình Linear Regressor
Mô hình Hồi quy Tuyến tính là mô hình đơn giản nhất: chỉ có **một lớp tuyến tính** (Linear Layer) duy nhất, **không có hàm kích hoạt phi tuyến** (activation function).

**Công thức:** $\hat{y} = X \cdot W + b$

Trong đó:
- $X$: Ma trận đầu vào (features) - kích thước `[batch_size, 5]`
- $W$: Ma trận trọng số (weights) - kích thước `[5, 1]`  
- $b$: Vector độ lệch (bias) - kích thước `[1]`
- $\hat{y}$: Giá trị dự đoán (predicted value) - kích thước `[batch_size, 1]`

In [4]:
class LinearRegressor(nn.Module):
    """Mô hình Hồi quy Tuyến tính: y = X @ W + b"""
    
    def __init__(self, input_dim):
        super().__init__()
        # Chỉ một lớp Linear duy nhất: input_dim features -> 1 output
        self.linear = nn.Linear(input_dim, 1)
    
    def forward(self, x):
        return self.linear(x)

# Khởi tạo mô hình
input_dim = tX_train.shape[1]
model = LinearRegressor(input_dim).to(DEVICE)
print(model)
print(f"\nTổng số tham số: {sum(p.numel() for p in model.parameters())}")
print(f"  - Weights (W): {model.linear.weight.shape} = {model.linear.weight.numel()} tham số")
print(f"  - Bias (b):    {model.linear.bias.shape}    = {model.linear.bias.numel()} tham số")

LinearRegressor(
  (linear): Linear(in_features=5, out_features=1, bias=True)
)

Tổng số tham số: 6
  - Weights (W): torch.Size([1, 5]) = 5 tham số
  - Bias (b):    torch.Size([1])    = 1 tham số


## 5. Training Loop (Vòng lặp huấn luyện)

### Quy trình huấn luyện mỗi epoch:
1. **Forward pass**: Tính $\hat{y} = X \cdot W + b$
2. **Tính loss**: $\text{MSE} = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$
3. **Backward pass**: Tính gradient $\frac{\partial L}{\partial W}$, $\frac{\partial L}{\partial b}$
4. **Cập nhật tham số**: $W \leftarrow W - \eta \cdot \frac{\partial L}{\partial W}$

Sử dụng **Early Stopping** dựa trên validation loss để tránh overfitting.

In [5]:
# === Hàm loss và Optimizer ===
criterion = nn.MSELoss()  # Mean Squared Error
optimizer = torch.optim.SGD(model.parameters(), lr=LR)  # Stochastic Gradient Descent

def train_epoch(model, loader, optimizer, criterion, device):
    """Huấn luyện 1 epoch."""
    model.train()
    running_loss = 0.0
    n = 0
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        
        # Forward pass
        pred = model(xb)
        loss = criterion(pred, yb)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        
        # Cập nhật tham số
        optimizer.step()
        
        bs = xb.size(0)
        running_loss += loss.item() * bs
        n += bs
    return running_loss / n

def eval_loss(model, loader, criterion, device):
    """Đánh giá loss trên tập validation/test."""
    model.eval()
    running_loss = 0.0
    n = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            pred = model(xb)
            loss = criterion(pred, yb)
            bs = xb.size(0)
            running_loss += loss.item() * bs
            n += bs
    return running_loss / n

# === Training Loop ===
best_val = float("inf")
best_state = None
history = {"train_loss": [], "val_loss": []}

for epoch in range(1, EPOCHS + 1):
    tr_loss  = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss = eval_loss(model, val_loader, criterion, DEVICE)
    
    history["train_loss"].append(tr_loss)
    history["val_loss"].append(val_loss)
    
    # Lưu mô hình tốt nhất dựa trên validation loss
    if val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}
    
    # In kết quả mỗi 20 epochs
    if epoch % 20 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d}  train_loss={tr_loss:.6f}  val_loss={val_loss:.6f}")

# Load lại mô hình tốt nhất
if best_state is not None:
    model.load_state_dict(best_state)
print(f"\nBest validation loss: {best_val:.6f}")

Epoch 001  train_loss=0.534157  val_loss=0.391453
Epoch 020  train_loss=0.002458  val_loss=0.001781
Epoch 040  train_loss=0.000009  val_loss=0.000007
Epoch 060  train_loss=0.000000  val_loss=0.000000
Epoch 080  train_loss=0.000000  val_loss=0.000000
Epoch 100  train_loss=0.000000  val_loss=0.000000
Epoch 120  train_loss=0.000000  val_loss=0.000000
Epoch 140  train_loss=0.000000  val_loss=0.000000
Epoch 160  train_loss=0.000000  val_loss=0.000000
Epoch 180  train_loss=0.000000  val_loss=0.000000
Epoch 200  train_loss=0.000000  val_loss=0.000000

Best validation loss: 0.000000


## 6. Đánh giá mô hình trên tập Test
Sử dụng 4 chỉ số đánh giá:
- **MSE** (Mean Squared Error): Trung bình bình phương sai số
- **MAE** (Mean Absolute Error): Trung bình trị tuyệt đối sai số  
- **RMSE** (Root MSE): Căn bậc hai của MSE
- **R² Score**: Hệ số xác định — càng gần 1 càng tốt

In [6]:
def evaluate_full(model, loader, device):
    """Đánh giá đầy đủ mô hình: MSE, MAE, RMSE, R²."""
    model.eval()
    ys = []
    preds = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            pred = model(xb).cpu().numpy().reshape(-1)
            preds.append(pred)
            ys.append(yb.numpy().reshape(-1))
    
    y_true = np.concatenate(ys)
    y_pred = np.concatenate(preds)
    
    mse  = np.mean((y_true - y_pred) ** 2)
    mae  = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(mse)
    
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot != 0 else float("nan")
    
    return {"MSE": mse, "MAE": mae, "RMSE": rmse, "R2": r2}, y_true, y_pred

results, y_true, y_pred = evaluate_full(model, test_loader, DEVICE)

print("="*50)
print("  KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST")
print("="*50)
for metric, value in results.items():
    print(f"  {metric:>5s}: {value:.6f}")
print("="*50)

  KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST
    MSE: 0.000000
    MAE: 0.000018
   RMSE: 0.000023
     R2: 1.000000


## 7. Trọng số đã học được
Kiểm tra trọng số (weights) và bias mà mô hình đã học — cho biết mức độ ảnh hưởng của từng feature đến giá mua xe.

In [7]:
# Lấy tên các cột features
feature_names = list(df_train.columns[:-1])

# Lấy trọng số đã học
weights = model.linear.weight.detach().cpu().numpy().flatten()
bias    = model.linear.bias.detach().cpu().numpy().item()

print("Trọng số (Weights) đã học:")
print("-" * 40)
for name, w in zip(feature_names, weights):
    print(f"  {name:>20s}: {w:+.6f}")
print(f"{'Bias':>22s}: {bias:+.6f}")
print("-" * 40)

Trọng số (Weights) đã học:
----------------------------------------
                gender: -0.000001
                   age: +0.095271
         annual Salary: +0.094158
      credit card debt: -0.000001
             net worth: +0.071583
                  Bias: +0.492198
----------------------------------------


## 8. Biểu đồ Loss theo Epoch

In [8]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(history["train_loss"], label="Train Loss", color="#2196F3", linewidth=2)
plt.plot(history["val_loss"],   label="Validation Loss", color="#FF5722", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Linear Regression - Training & Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Biểu đồ So sánh Giá trị Thực tế vs Dự đoán

In [9]:
plt.figure(figsize=(8, 8))
plt.scatter(y_true, y_pred, alpha=0.7, color="#4CAF50", edgecolors="white", s=80)
plt.plot([0, 1], [0, 1], 'r--', linewidth=2, label="Đường lý tưởng (y = x)")
plt.xlabel("Giá trị thực tế (Actual)")
plt.ylabel("Giá trị dự đoán (Predicted)")
plt.title(f"Linear Regression: Actual vs Predicted (R² = {results['R2']:.4f})")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()